# PyCaret Experiments

This notebook executes all PyCaret experiments used in this thesis.

Configuration:
- Datasets: Breast Cancer, Wine, Titanic
- Seeds: 42, 123, 2026
- 5-fold Stratified Cross-Validation
- 10-minute search budget
- Macro F1 used for model selection
- Evaluation on an external untouched test set

In [2]:
from src.pycaret_experiments import run_all_pycaret_experiments

results = run_all_pycaret_experiments()

results


Running PyCaret: breast_cancer | seed 42
Best model: LogisticRegression
Test Macro F1: 0.9812
Search runtime: 4.49 seconds

Running PyCaret: breast_cancer | seed 123
Best model: LogisticRegression
Test Macro F1: 0.9716
Search runtime: 5.59 seconds

Running PyCaret: breast_cancer | seed 2026
Best model: LogisticRegression
Test Macro F1: 0.9716
Search runtime: 6.67 seconds

Running PyCaret: wine | seed 42
Best model: LogisticRegression
Test Macro F1: 0.971
Search runtime: 6.18 seconds

Running PyCaret: wine | seed 123
Best model: LogisticRegression
Test Macro F1: 1.0
Search runtime: 6.02 seconds

Running PyCaret: wine | seed 2026
Best model: LogisticRegression
Test Macro F1: 0.9743
Search runtime: 7.3 seconds

Running PyCaret: titanic | seed 42
Best model: LGBMClassifier
Test Macro F1: 0.7978
Search runtime: 6.79 seconds

Running PyCaret: titanic | seed 123
Best model: LGBMClassifier
Test Macro F1: 0.7452
Search runtime: 5.86 seconds

Running PyCaret: titanic | seed 2026
Best model: Gra

,framework,dataset,seed,best_model,accuracy,precision_macro,recall_macro,f1_macro,search_runtime_seconds,total_runtime_seconds
0,PyCaret,breast_cancer,42,LogisticRegression,0.982456,0.981151,0.981151,0.981151,4.490112,5.365379
1,PyCaret,breast_cancer,123,LogisticRegression,0.973684,0.974106,0.969246,0.971583,5.586567,6.292951
2,PyCaret,breast_cancer,2026,LogisticRegression,0.973684,0.974106,0.969246,0.971583,6.672980,9.161379
3,PyCaret,wine,42,LogisticRegression,0.972222,0.977778,0.966667,0.970962,6.181032,8.640538
4,PyCaret,wine,123,LogisticRegression,1.000000,1.000000,1.000000,1.000000,6.024585,8.432649
5,PyCaret,wine,2026,LogisticRegression,0.972222,0.974359,0.976190,0.974321,7.296287,9.199281
6,PyCaret,titanic,42,LGBMClassifier,0.809160,0.797840,0.797840,0.797840,6.790292,8.874242
7,PyCaret,titanic,123,LGBMClassifier,0.763359,0.750320,0.741667,0.745168,5.862944,7.657188
8,PyCaret,titanic,2026,GradientBoostingClassifier,0.797710,0.793801,0.769444,0.777479,5.228045,7.089507


In [ ]:
import time
from pathlib import Path

import pandas as pd

from pycaret.classification import ClassificationExperiment

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

PROCESSED_ROOT = PROJECT_ROOT / "Processed"
RESULTS_ROOT = PROJECT_ROOT / "Results"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = [
    "breast_cancer",
    "wine",
    "titanic"
]

SEEDS = [42, 123, 2026]

TIME_BUDGET_MINUTES = 10
CV_FOLDS = 5

In [ ]:
print(PROCESSED_ROOT)
print(PROCESSED_ROOT.exists())

print(RESULTS_ROOT)
print(RESULTS_ROOT.exists())

c:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed
True
c:\Users\souha\Downloads\Human Centered AutoML Thesis\Results
True


In [ ]:
print(PROCESSED_ROOT.resolve())
print(PROCESSED_ROOT.exists())

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = [
    "breast_cancer",
    "wine",
    "titanic"
]

SEEDS = [42, 123, 2026]

TIME_BUDGET_MINUTES = 10
CV_FOLDS = 5

C:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed
True


In [ ]:
def run_pycaret_experiment(dataset_name, seed):
    """
    Run one PyCaret experiment using:
    - externally preprocessed data
    - 5-fold stratified cross-validation
    - a 10-minute model-comparison budget
    - Macro F1 for model selection
    - one final evaluation on the untouched test set
    """

    print(f"\nRunning PyCaret: {dataset_name} | seed {seed}")

    folder = PROCESSED_ROOT / dataset_name / f"seed_{seed}"

    # Load the exact train/test split created previously
    X_train = pd.read_csv(folder / "X_train.csv")
    X_test = pd.read_csv(folder / "X_test.csv")

    y_train = pd.read_csv(folder / "y_train.csv").squeeze("columns")
    y_test = pd.read_csv(folder / "y_test.csv").squeeze("columns")

    # PyCaret expects the target inside the DataFrame
    train_data = X_train.copy()
    train_data["target"] = y_train.to_numpy()

    test_data = X_test.copy()
    test_data["target"] = y_test.to_numpy()

    experiment = ClassificationExperiment()

    total_start = time.perf_counter()

    experiment.setup(
        data=train_data,
        target="target",

        # Give PyCaret the external untouched test set
        test_data=test_data,

        # The data is already imputed, encoded and standardized
        preprocess=False,

        fold_strategy="stratifiedkfold",
        fold=CV_FOLDS,
        fold_shuffle=True,

        session_id=seed,

        n_jobs=-1,
        verbose=False
    )

    # PyCaret's default F1 is not sufficient for our multiclass
    # methodology, so we explicitly create Macro F1.
    experiment.add_metric(
        id="macro_f1",
        name="Macro F1",
        score_func=f1_score,
        greater_is_better=True,
        multiclass=True,
        average="macro"
    )

    search_start = time.perf_counter()

    best_model = experiment.compare_models(
        sort="Macro F1",
        fold=CV_FOLDS,
        budget_time=TIME_BUDGET_MINUTES,
        turbo=True,
        errors="ignore",
        verbose=False
    )

    search_runtime_seconds = time.perf_counter() - search_start

    # Store the cross-validation leaderboard
    leaderboard = experiment.pull().copy()

    # Predict only after model selection is finished
    predictions = experiment.predict_model(
        best_model,
        data=X_test,
        verbose=False
    )

    y_pred = predictions["prediction_label"]

    total_runtime_seconds = time.perf_counter() - total_start

    result = {
        "framework": "PyCaret",
        "dataset": dataset_name,
        "seed": seed,
        "best_model": type(best_model).__name__,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "recall_macro": recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "f1_macro": f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "search_runtime_seconds": search_runtime_seconds,
        "total_runtime_seconds": total_runtime_seconds
    }

    # Save the leaderboard from this individual experiment
    leaderboard_path = (
        RESULTS_ROOT
        / f"pycaret_leaderboard_{dataset_name}_seed_{seed}.csv"
    )

    leaderboard.to_csv(leaderboard_path, index=False)

    print("Best model:", result["best_model"])
    print("Test Macro F1:", round(result["f1_macro"], 4))
    print("Search runtime:", round(search_runtime_seconds, 2), "seconds")

    return result

In [ ]:
print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("Resolved path:", PROCESSED_ROOT.resolve())

folder = PROCESSED_ROOT / "breast_cancer" / "seed_42"

print("Experiment folder:", folder.resolve())
print("Folder exists:", folder.exists())

if folder.exists():
    print(list(folder.iterdir()))

PROCESSED_ROOT: ..\Processed
Resolved path: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed
Experiment folder: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed\breast_cancer\seed_42
Folder exists: False


In [ ]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("Processed output will be:", (Path.cwd().parent / "Processed").resolve())

Current folder: c:\Users\souha\Downloads\Human Centered AutoML Thesis\Notebooks
Processed output will be: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed


In [ ]:
processed_splits = {}

In [ ]:
print(datasets.keys())

NameError: name 'datasets' is not defined

In [ ]:
print(processed_splits.keys())

dict_keys([])


In [ ]:
from pathlib import Path

project_root = Path.cwd().parent
output_root = project_root / "Processed"

output_root.mkdir(parents=True, exist_ok=True)

for dataset_name, dataset_splits in processed_splits.items():
    for seed, split in dataset_splits.items():

        output_dir = output_root / dataset_name / f"seed_{seed}"
        output_dir.mkdir(parents=True, exist_ok=True)

        split["X_train"].to_csv(output_dir / "X_train.csv", index=False)
        split["X_test"].to_csv(output_dir / "X_test.csv", index=False)
        split["y_train"].to_csv(output_dir / "y_train.csv", index=False)
        split["y_test"].to_csv(output_dir / "y_test.csv", index=False)

        print(f"Saved to: {output_dir.resolve()}")

NameError: name 'processed_splits' is not defined

In [ ]:
test_result = run_pycaret_experiment(
    dataset_name="breast_cancer",
    seed=42
)


Running PyCaret: breast_cancer | seed 42


FileNotFoundError: [Errno 2] No such file or directory: '..\\Processed\\breast_cancer\\seed_42\\X_train.csv'